# InstaWell Toy Example

A minimal example demonstrating the complete InstaWell pipeline using a tiny synthetic dataset.

## Dataset Description

- **Protein**: ProteinA
- **Ligand**: ATP (dose-response)
- **Concentrations**: 0, 1, 10, 100, 1000 µM (5 concentrations)
- **Replicates**: 2 technical replicates per condition
- **Controls**: NPC (non-protein control) at each concentration (~10× lower signal)
- **Temperature range**: 25-75°C in 5°C steps
- **Total wells**: 20 (10 rows × 2 columns)

## Expected Results

ProteinA shows thermal stabilization by ATP in a dose-dependent manner:
- **Apo (0 µM)**: Tm ≈ 45°C
- **1 µM ATP**: Tm ≈ 47°C
- **10 µM ATP**: Tm ≈ 52°C
- **100 µM ATP**: Tm ≈ 58°C
- **1000 µM ATP**: Tm ≈ 63°C (saturated)


In [ ]:
# Import the pipeline functions
from instawell import (
    setup_experiment,
    ingest_data,
    filter_wells,
    average_across_replicates,
    subtract_background,
    min_max_scale,
    calculate_derivative,
    find_min_temperature,
    calculate_curve_params,
)

import pandas as pd

## Step 0: Setup Experiment

Create the experiment directory and initialize the context.

In [ ]:
ctx = setup_experiment(
    experiment_name="toy_example",
    experiments_root="experiments",
    raw_data_path="toy_example_raw.csv",
    layout_data_path="toy_example_layout.csv",
    condition_fields=("concentration", "ligand", "protein", "buffer"),
    condition_separator="_",
    empty_condition_placeholder="0",
    non_protein_control_marker="NPC",
)

print(f"Experiment directory: {ctx.experiment_dir}")

## Step 1: Ingest and Organize Data

Parse the layout and organize raw data into long format.

In [ ]:
ingest_data(ctx)

# Preview the organized data
raw_organized = pd.read_csv(ctx.experiment_dir / "01_raw_organized_data.csv")
print(f"Shape: {raw_organized.shape}")
raw_organized.head(10)

## Step 2: Filter Wells (Optional)

In this toy example, all wells are good, so we don't filter any. Remeber we still have to run this step to record that no wells were filtered!

In [ ]:
filter_wells(ctx, wells_to_filter=[])  # No wells to filter

## Step 3: Average Replicates

Average the 2 technical replicates for each condition.

In [ ]:
average_across_replicates(ctx)

# Preview averaged data (wide format)
averaged = pd.read_csv(ctx.experiment_dir / "03_averaged_data.csv")
print(f"Shape: {averaged.shape}")
averaged.head()

## Step 4: Subtract Background (NPC)

Subtract the NPC (non-protein control) signal from each matching condition.

In [ ]:
subtract_background(ctx)

# Preview background-subtracted data
bg_sub = pd.read_csv(ctx.experiment_dir / "04_bg_subtracted_data.csv")
print(f"Shape: {bg_sub.shape}")
print(f"\nColumns (NPC removed): {list(bg_sub.columns)}")
bg_sub.head()

## Step 5: Min-Max Scale

Normalize each condition to [0, 1] for easier comparison.

In [ ]:
min_max_scale(ctx)

scaled = pd.read_csv(ctx.experiment_dir / "05_min_max_scaled_data.csv")
scaled.head()

## Step 6: Calculate Derivative

Compute the negative derivative to find inflection points (Tm).

In [ ]:
calculate_derivative(ctx)

deriv = pd.read_csv(ctx.experiment_dir / "06_derivative_data.csv")
deriv.head()

## Step 7: Find Minimum Temperature (Tm)

Extract the Tm for each condition.

In [ ]:
find_min_temperature(ctx)

# View the Tm results
min_temps = pd.read_csv(ctx.experiment_dir / "07_min_temperatures.csv")
print("\n📊 Melting Temperatures (Tm):")
print(min_temps[['concentration', 'ligand', 'protein', 'min_temperature']].to_string(index=False))

## Step 8: Fit 4PL Dose-Response Curve

Fit a 4-parameter logistic model to the Tm vs concentration data.

In [ ]:
calculate_curve_params(ctx, weighting="none")

# View curve parameters
curve_params = pd.read_csv(ctx.experiment_dir / "08_curve_params.csv")
print("\n📈 4PL Curve Parameters:")
print(f"Bottom (Tm at 0 µM): {curve_params['bottom'].values[0]:.2f}°C")
print(f"Top (Tm at ∞ µM): {curve_params['top'].values[0]:.2f}°C")
print(f"EC50: {curve_params['EC50'].values[0]:.2f} µM")
print(f"Hill slope: {curve_params['Hill'].values[0]:.2f}")
print(f"\nFull parameters:")
curve_params[['ligand', 'protein', 'buffer', 'bottom', 'top', 'EC50', 'Hill', 'rmse']]

## Visualize with Jupyter Widgets

Use InstaWell's interactive widgets to browse the results.

In [ ]:
from instawell import raw_figures_widget, processed_figures_widget, min_temp_figures_widget

# View raw per-well traces
raw_figures_widget(ctx)

In [ ]:
# View processed/averaged traces
processed_figures_widget(ctx, data_source="bg_subtracted_data", color_scale_mapping="Viridis")

In [ ]:
# View Tm scatter plot with dose-response fit
min_temp_figures_widget(ctx, mode="log10_fit", color_scale="Thermal")

## Summary

✅ Successfully processed a complete thermal shift assay dataset

✅ Identified dose-dependent thermal stabilization of ProteinA by ATP

✅ Extracted Tm values and fitted a 4PL dose-response curve

All output files are saved in: `experiments/toy_example/`